# LILA-Pioneer exploratory noise budget

This notebook builds an equivalent differential-displacement ASD model for the proposed 5 km LILA-Pioneer single-arm strainmeters. It keeps published LILA terms visually distinct from literature-transferred alternatives, assumption-based estimates, and diagnostic upper limits.

**Visual encoding**

- Dark solid: directly specified or evaluated from the LILA noise paper
- Pale solid: published alternative with weaker connection to final hardware
- Dashed: assumption-based estimate or allocation without a measured spectrum
- Dotted: diagnostic upper limit that is not included in the total

The notebook is intentionally transparent: edit `LILAParameters` rather than changing equations throughout the analysis.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from lila_pioneer_noise_budget import (
    CURVE_STATUS,
    LIMITATIONS,
    LILAParameters,
    assumptions_table,
    calculate_noise_curves,
    curves_dataframe,
    frequency_grid,
    plot_noise_budget,
    reference_table,
)


## 1. Parameters

Published LILA inputs and editable assumptions are kept together. The most important outstanding inputs are the received optical power/link budget, actual coating/substrate/mount materials, laser stabilization spectrum, amplitude-to-phase coupling, alignment noise, local thermal stability, electronics phase noise, and lunar differential seismic coherence.


In [ ]:
p = LILAParameters()

# Examples of sensitivity studies:
# p.effective_detected_power_W = 0.100
# p.amplitude_to_phase_rad_per_rin = 1e-5
# p.temperature_asd_1Hz_K_per_rtHz = 1e-8
# p.assumed_differential_seismic_coupling = 1e-3

assumptions_table(p)


## 2. Calculate and plot the noise budget


In [ ]:
f = frequency_grid(p)
curves = calculate_noise_curves(f, p)
fig = plot_noise_budget(f, curves)
plt.show()


## 3. Reference values

All values below are displacement ASDs in m/√Hz. The conservative thermal curve is an alternative case, not an additional term in the total.


In [ ]:
table = reference_table(f, curves)
table.style.format("{:.3e}")


## 4. Curve provenance/status


In [ ]:
pd.DataFrame(
    [{"curve": name, "status": status} for name, status in CURVE_STATUS.items()]
)


## 5. Known limitations and improvement path


In [ ]:
pd.DataFrame(
    {"priority": range(1, len(LIMITATIONS) + 1), "limitation_or_needed_improvement": LIMITATIONS}
)


## 6. Export the current model realization


In [ ]:
output_dir = Path("lila_noise_outputs")
output_dir.mkdir(exist_ok=True)

plot_noise_budget(f, curves, output_dir / "lila_pioneer_noise_budget.png")
plt.close("all")
curves_dataframe(f, curves).to_csv(output_dir / "lila_pioneer_noise_budget_data.csv", index=False)
reference_table(f, curves).to_csv(output_dir / "lila_pioneer_reference_values.csv")
assumptions_table(p).to_csv(output_dir / "lila_pioneer_model_assumptions.csv", index=False)

print(f"Saved outputs to {output_dir.resolve()}")


## References

1. T. Creighton et al., *Fundamental Noise and Gravitational-Wave Sensitivity of the Laser Interferometer Lunar Antenna (LILA)*, arXiv:2508.18437.
2. K. Jani et al., *Laser Interferometer Lunar Antenna (LILA): Advancing the U.S. Priorities in Gravitational-wave and Lunar Science*, arXiv:2508.11631.
3. J. B. Camp et al., *Analysis of light noise sources in a recycled Michelson interferometer with Fabry–Perot arms*, JOSA A 17, 120–128 (2000).
4. G. Harry, *Coating thermal noise formulas*, LIGO-T0900161.
5. S. Gras et al., Phys. Rev. D 95, 022001 (2017).
